In [ ]:
# === 1. Comprehensive Library Installation (CUDA 12.8 Compatible) ===
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --force-reinstall -q
!pip install --upgrade transformers accelerate bitsandbytes Pillow requests apify-client sentencepiece pandas -q

# === 2. Verifying Installation ===
import torch, bitsandbytes as bnb
print(f" PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f" GPU: {torch.cuda.get_device_name(0)}")
print(f" bitsandbytes: {bnb.__version__}")
print("\n Installation complete! If RAM is still full, click Restart from the top menu so the updates take effect in Python.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_DIR = "/content/drive/MyDrive/hajj_real_data"
os.makedirs(DRIVE_DIR, exist_ok=True)

IMAGES_DIR = os.path.join(DRIVE_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

CSV_PATH = os.path.join(DRIVE_DIR, "real_data.csv")

print(f"[OK] drive connected")
print(f"[OK] csv path: {CSV_PATH}")
print(f"[OK] images path: {IMAGES_DIR}")


In [ ]:
# Ensure torch + torchvision are working together
import torch, torchvision
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

print(f" PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f" torchvision: {torchvision.__version__}")

# Ensure transformers + LLaVA are working
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, pipeline
print(" LlavaForConditionalGeneration loaded!")
import torch
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f" bitsandbytes: {bnb.__version__}")

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
print(" BitsAndBytesConfig is ready!")
print(" Everything is set - Start loading the model!")


In [ ]:
import gc, torch, os
from PIL import Image as PILImage
from transformers import LlavaForConditionalGeneration, AutoProcessor, pipeline, BitsAndBytesConfig
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print(" (1/2) Loading LLaVA 1.5 7B 4bit...")
processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
llava_model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
print(" LLaVA ready in 4-bit!")

print(" (2/2) Loading NLLB for translation...")


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from PIL import Image as PILImage
nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
nllb_model_trans = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
print(" NLLB ready and no memory issues!")

print(" Ready - Run the Script Cell!")


In [ ]:
import torch

if torch.cuda.is_available():
    t = torch.cuda.get_device_properties(0).total_memory
    r = torch.cuda.memory_reserved(0)
    a = torch.cuda.memory_allocated(0)

    # Free space for Tensor allocations
    f = t - r

    print(f" 1. Total GPU Memory: {t / 1024**3:.2f} GB")
    print(f" 2. Active PyTorch Memory (Models): {a / 1024**3:.2f} GB")
    print(f" 3. Reserved but Unused (Ghost Memory): {(r - a) / 1024**3:.2f} GB")
    print(f" 4. Absolutely Free Space: {f / 1024**3:.2f} GB")
else:
    print("GPU not connected!")


In [ ]:
import gc

def translate_to_arabic(text):
    if not text.strip(): return ""
    inputs = nllb_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    translated = nllb_model_trans.generate(
        **inputs,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("arb_Arab"),
        max_new_tokens=300,
        num_beams=4
    )
    return nllb_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

AD_PROMPT = """USER: <image>
You are analyzing an Egyptian advertisement for Hajj pilgrimage, Umrah, or Saudi visa services.
Treat this as a MARKETING ANALYST, not a visual descriptor.

Answer these 3 questions:
1. WHAT IS BEING SOLD? (e.g. Hajj package, visa service, hotel booking, Umrah trip)
2. KEY DETAILS: What specific information is advertised? (prices, season/year like 1447H, package type, company name, phone number, offers)
3. CALL TO ACTION: What does this ad want people to do?

IMPORTANT: Ignore passport photos or ID card photos of customers if present - focus only on the advertisement message.
ASSISTANT:"""

def print_memory_status(ad_id=""):
    if torch.cuda.is_available():
        t = torch.cuda.get_device_properties(0).total_memory
        r = torch.cuda.memory_reserved(0)
        free_gb = (t - r) / 1024**3
        return f"[VRAM F: {free_gb:.1f} GB]"
    return ""

def generate_image_description(image_path):
    img = PILImage.open(image_path).convert("RGB")
    inputs = processor(text=AD_PROMPT, images=img, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = llava_model.generate(**inputs, max_new_tokens=250, do_sample=False)

    # Extract text in English
    en_full = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

    # ----- Immediate variable cleanup for memory efficiency -----
    del inputs
    del output
    img.close()

    # Mandatory memory garbage collection
    gc.collect()
    torch.cuda.empty_cache()
    # -----------------------------------------------------------

    max_len = 400
    if len(en_full) <= max_len:
        ar = translate_to_arabic(en_full)
    else:
        parts = [en_full[i:i+max_len] for i in range(0, len(en_full), max_len)]
        ar = " ".join([translate_to_arabic(p) for p in parts])

    ar_clean = ar.replace("\n", " ").strip()
    if not ar_clean.startswith("صورة"):
        ar_clean = "صورة " + ar_clean

    mem_stat = print_memory_status()
    print(f"  -> Analyzed {image_path.split('/')[-1]} {mem_stat}")

    return ar_clean

print("  Updated: Immediate variable cleanup enabled + RAM monitoring active!")


In [ ]:
import pandas as pd
import os

CSV_PATH   = "/content/drive/MyDrive/hajj_real_data/real_data.csv"
IMAGES_DIR = "/content/drive/MyDrive/hajj_real_data/images"

df = pd.read_csv(CSV_PATH)

bad_desc_mask = df["Image Description"] == "وصف الصورة المحلل آليا (غير متاح حاليا)"
bad_rows = df[bad_desc_mask]
print(f" Found {len(bad_rows)} ads to fix.")

fixed_count = 0
for idx, row in bad_rows.iterrows():
    ad_id = row["Sample ID"]
    img_path = os.path.join(IMAGES_DIR, f"{ad_id}.jpg")

    if os.path.exists(img_path):
        print(f"Fixing {ad_id}...")
        try:
            new_desc = generate_image_description(img_path)
            df.at[idx, "Image Description"] = new_desc
            fixed_count += 1
        except Exception as e:
            print(f" Failed to fix {ad_id}: {e}")
    else:
        print(f" Image not found for {ad_id} ({img_path})")

if fixed_count > 0:
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(f" Successfully fixed and saved {fixed_count} ads!")


In [ ]:
import requests, time, os, csv, urllib.parse
import pandas as pd
from datetime import datetime
from IPython.display import display, clear_output

# ===========================================================
#  Settings
# ===========================================================
DRIVE_DIR    = "/content/drive/MyDrive/hajj_real_data"
IMAGES_DIR   = os.path.join(DRIVE_DIR, "images")
CSV_PATH     = os.path.join(DRIVE_DIR, "real_data.csv")
LOG_PATH     = os.path.join(DRIVE_DIR, "collection_log.txt")
TOTAL_TARGET = 2000
os.makedirs(IMAGES_DIR, exist_ok=True)

# ===========================================================
#  API Tokens (5 working tokens)
# ===========================================================
API_TOKENS = [
"Your_Token"
]
current_token_idx = 0

# ===========================================================
#  Search Groups
# ===========================================================
QUERY_GROUPS = [
    # -- Group 1 -------------------------------------------
    ["باقة حج", "حملة حج", "رحلة حج", "حج 1447", "حج مفرد"],
    # -- Group 2 -------------------------------------------
    ["رحلة عمرة", "باقة عمرة", "حجز عمرة", "عمرة 1447", "عمرة جماعية"],
    # -- Group 3 -------------------------------------------
    ["عمرة رمضان", "عمرة مميزة", "عمرة شاملة", "برنامج عمرة", "موسم عمرة"],
    # -- Group 4 -------------------------------------------
    ["تأشيرة عمرة", "تأشيرة حج", "فيزا عمرة", "فيزا حج", "تصريح حج"],
    # -- Group 5 -------------------------------------------
    ["حملة عمرة", "مكتب حج", "شركة عمرة", "وكالة حج", "مؤسسة حج"],
    # -- Group 6 -------------------------------------------
    ["فندق مكة", "سكن معتمرين", "فنادق عمرة", "سكن حجاج", "إقامة مكة"],
    # -- Group 7 -------------------------------------------
    ["طيران عمرة", "طيران حج", "خدمات حجاج", "مواصلات حج", "تنظيم عمرة"],
    # -- Group 8 -------------------------------------------
    ["سفرة عمرة", "رحلة مكة", "سفرة حج", "فيزا سعودية"],
    # -- Group 9 -------------------------------------------
    ["عروض عمرة", "أسعار عمرة", "عمرة اقتصادية", "حج التمتع", "حج القران"],
    # -- Group 10 ------------------------------------------
    ["زيارة المدينة", "فندق المدينة", "مكة المكرمة", "الحرم المكي", "المسجد النبوي"],
    # -- Group 11 & 12 -------------------------------------
    ["خدمات معتمرين", "مناسك حج", "باقات حج", "رحلات مكة", "السفر للحج"],
]

# -- Blacklist --
GENERAL_BLACKLIST = [
    "سيارة", "سيارات", "محرك", "مكينة", "قير", "جير", "ورشة", "زيت", "قطع غيار", "كراج", "بنشر",
    "توضيب", "دهان", "هيكل", "تشغيل سيارة", "أسنان", "اسنان", "تقويم", "زراعة أسنان", "عيادة",
    "طبيب", "دكتور", "مستشفى", "علاج", "تجميل أسنان", "بروتيز", "جسور", "تبييض", "طب الأسنان", "أشعة",
    "ليزر", "بوتوكس", "فيلر", "كيراتين", "مساج", "صالون", "سبا", "بيوتي", "تخسيس", "رجيم",
    "شقة", "شقق", "عقار", "عمارة", "بناء", "سيراميك", "مقاولات", "تشطيب", "ديكور", "أثاث",
    "موبايل", "لابتوب", "تليفون", "إنترنت", "شبكة", "فستان", "كوتش", "أحذية", "ملابس سواريه",
]
CAR_BLACKLIST = GENERAL_BLACKLIST

# -- Strong Phrases --
HAJJ_STRONG_PHRASES = [
    "رحلة عمرة", "رحله عمره", "باقة عمرة", "باقه عمره", "حجز عمرة", "حجز عمره", "عمرة جماعية", 
    "عمره جماعيه", "برنامج عمرة", "برنامج عمره", "موسم عمرة", "موسم عمره", "عمرة رمضان", 
    "عمره رمضان", "عمرة مميزة", "عمرة شاملة", "حملة عمرة", "حمله عمره", "عروض عمرة", 
    "أسعار عمرة", "عمرة اقتصادية", "سفرة عمرة", "سفره عمره", "فنادق عمرة", "تنظيم عمرة", 
    "طيران عمرة", "خدمات معتمرين", "سكن معتمرين", "تأشيرة عمرة", "فيزا عمرة", "رحلة حج", 
    "رحله حج", "باقة حج", "باقه حج", "حملة حج", "حمله حج", "مكتب حج", "وكالة حج", "مؤسسة حج", 
    "سفرة حج", "سفره حج", "طيران حج", "تصريح حج", "تأشيرة حج", "فيزا حج", "حج 1447", 
    "حج مفرد", "حج التمتع", "حج القران", "باقات حج", "مناسك حج", "خدمات حجاج", "مواصلات حج", 
    "سكن حجاج", "السفر للحج", "رحلات مكة", "مكة المكرمة", "المدينة المنورة", "المسجد النبوي", 
    "الحرم المكي", "البيت الحرام", "الكعبة المشرفة", "فندق مكة", "إقامة مكة", "زيارة المدينة", 
    "فندق المدينة",
]

# -- Single Keywords --
HAJJ_SINGLE_STRONG = [
    "معتمرين", "معتمر", "الحجاج", "حجاج", "طواف", "سعي", "منى", "عرفة", "مزدلفة", "مشعر",
    "الإحرام", "إحرام", "ملابس إحرام", "مستلزمات حج",
]

HAJJ_KEYWORDS = HAJJ_STRONG_PHRASES + HAJJ_SINGLE_STRONG

# ===========================================================
#  Targets
# ===========================================================
TARGETS = {
    "مصري":  {
        "countries": ["EG"],
        "target": 800
    },
    "سعودي": {
        "countries": ["SA"],
        "target": 800
    },
    "فصحى":  {
        "countries": [
            "AE", "KW", "QA", "BH", "OM", "JO", "LB", "PS",
            "DZ", "MA", "TN", "LY", "MR", "IQ", "SD", "YE",
        ],
        "target": 623
    },
}

fieldnames = ["Sample ID", "Advertisement Type", "Dialect", "Text Content",
              "Image Description", "Metadata", "Image File", "Label"]

# ===========================================================
#  Helper Functions
# ===========================================================
def is_hajj_related(text):
    t = text or ""
    if any(cw in t for cw in GENERAL_BLACKLIST):
        return False
    if any(ph in t for ph in HAJJ_STRONG_PHRASES):
        return True
    single_hits = sum(1 for kw in HAJJ_SINGLE_STRONG if kw in t)
    return single_hits >= 2

def log(msg, level="INFO"):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] [{level}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def print_status(existing_count, dialect_counts, session_saved, last_msg=""):
    clear_output(wait=True)
    pct    = existing_count / TOTAL_TARGET * 100
    filled = int(pct / 2)
    bar    = "#" * filled + "-" * (50 - filled)
    print(f"\n{'='*62}")
    print(f"  PROGRESS: [{bar}] {pct:.1f}%")
    print(f"  Total: {existing_count}/{TOTAL_TARGET} ads saved")
    print(f"{'='*62}")
    print(f"  masri  (EG): {dialect_counts['مصري']:>4} / {TARGETS['مصري']['target']}  {'[DONE]' if dialect_counts['مصري']>=TARGETS['مصري']['target'] else ''}")
    print(f"  saudi  (SA): {dialect_counts['سعودي']:>4} / {TARGETS['سعودي']['target']}  {'[DONE]' if dialect_counts['سعودي']>=TARGETS['سعودي']['target'] else ''}")
    print(f"  fusha (OTHER): {dialect_counts['فصحى']:>3} / {TARGETS['فصحى']['target']}  {'[DONE]' if dialect_counts['فصحى']>=TARGETS['فصحى']['target'] else ''}")
    print(f"{'='*62}")
    print(f"  session adds: {session_saved}  |  active token: #{current_token_idx+1}/{len(API_TOKENS)}")
    if last_msg:
        print(f"  last: {last_msg}")
    print(f"{'='*62}\n")

# ===========================================================
#  Apify Integration
# ===========================================================
def call_apify(url_template, method="GET", json_data=None):
    global current_token_idx
    while current_token_idx < len(API_TOKENS):
        token = API_TOKENS[current_token_idx]
        url   = url_template.format(token=token)
        try:
            r = requests.post(url, json=json_data, timeout=30) if method == "POST" \
                else requests.get(url, timeout=30)
            if r.status_code in [200, 201]: return r
            elif r.status_code in [400, 402, 403, 429]:
                log(f"token #{current_token_idx+1} failed, rotating...", "TOKEN")
                current_token_idx += 1
                time.sleep(2)
            else: return r
        except:
            current_token_idx += 1
            time.sleep(3)
    return None

def fetch_ads(country_code, queries_list, max_items):
    urls = [{"url": f"https://www.facebook.com/ads/library/?active_status=all&ad_type=all&country={country_code}&q={urllib.parse.quote(q)}&search_type=keyword_unordered&media_type=all&start_date[min]=2025-01-01&start_date[max]=2026-12-31"} for q in queries_list]
    log(f"FETCH | {country_code} | queries={len(queries_list)}", "FETCH")
    run_req = call_apify("https://api.apify.com/v2/acts/curious_coder~facebook-ads-library-scraper/runs?token={token}", method="POST", json_data={"urls": urls, "maxItems": max_items})
    if not run_req or run_req.status_code != 201: return []
    run_id = run_req.json()["data"]["id"]
    while True:
        sr = call_apify(f"https://api.apify.com/v2/actor-runs/{run_id}?token={{token}}")
        if not sr: return []
        status = sr.json()["data"]["status"]
        if status in ["SUCCEEDED", "FAILED", "ABORTED"]: break
        time.sleep(6)
    ir = call_apify(f"https://api.apify.com/v2/actor-runs/{run_id}/dataset/items?token={{token}}")
    return ir.json() if ir else []

# ===========================================================
#  Classification & Download
# ===========================================================
def detect_ad_type(text):
    text = text or ""
    if any(kw in text for kw in ["تأشيرة", "فيزا"]): return "visa_service"
    elif any(kw in text for kw in ["عمرة", "عمره"]): return "umrah_package"
    elif any(kw in text for kw in ["حج", "باقة", "حملة"]): return "hajj_package"
    elif any(kw in text for kw in ["فندق", "سكن", "إقامة"]): return "accommodation"
    else: return "islamic_travel"

def download_image(url, save_path):
    response = requests.get(url, stream=True, timeout=15)
    response.raise_for_status()
    with open(save_path, "wb") as f:
        for chunk in response.iter_content(1024): f.write(chunk)

# ===========================================================
#  Checkpoint
# ===========================================================
dialect_counts = {"مصري": 0, "سعودي": 0, "فصحى": 0}
country_counts = {}
existing_texts = set()
existing_count = 0

import re
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    existing_count = len(df)
    existing_texts = set(df["Text Content"].dropna().str.replace("\n", " ").str.strip())
    for _, row in df.iterrows():
        d = str(row.get("Dialect", ""))
        if d in dialect_counts: dialect_counts[d] += 1
        m = str(row.get("Metadata", ""))
        c_match = re.search(r"country:\s*([A-Za-z]{2})", m)
        if c_match:
            c = c_match.group(1).upper()
            country_counts[c] = country_counts.get(c, 0) + 1
    print_status(existing_count, dialect_counts, 0, "checkpoint loaded")
else:
    with open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

collection_date = datetime.now().strftime("%Y-%m-%d")
session_saved   = 0

# ===========================================================
#  Main Execution
# ===========================================================
for dialect_name, info in TARGETS.items():
    max_target = info["target"]
    countries  = info["countries"]
    if dialect_counts[dialect_name] >= max_target: continue

    for country_code in countries:
        if dialect_counts[dialect_name] >= max_target: break
        if dialect_name == "فصحى" and country_counts.get(country_code, 0) >= 100: continue

        group_idx = 0
        empty_rounds = 0
        while dialect_counts[dialect_name] < max_target:
            if current_token_idx >= len(API_TOKENS): break
            needed = max_target - dialect_counts[dialect_name]
            current_grp = QUERY_GROUPS[group_idx % len(QUERY_GROUPS)]
            fetch_count = min(needed + 100, 400)

            print_status(existing_count, dialect_counts, session_saved, f"fetching {country_code}")
            ads_raw = fetch_ads(country_code, current_grp, fetch_count)

            batch_added = 0
            for ad in (ads_raw or []):
                if dialect_counts[dialect_name] >= max_target: break
                snap = ad.get("snapshot") or {}
                text_parts = []
                if (snap.get("body") or {}).get("text"): text_parts.append(snap["body"]["text"])
                if snap.get("title"): text_parts.append(snap["title"])
                page_name = ad.get("page_name") or ""
                check_text = (" ".join(text_parts) + " " + page_name).strip()
                clean_text = " ".join(text_parts).replace("\n", " ").strip()

                if not check_text or not is_hajj_related(check_text): continue
                if clean_text in existing_texts: continue

                image_url = None
                if snap.get("images"): image_url = snap["images"][0].get("resized_image_url") or snap["images"][0].get("original_image_url")
                elif snap.get("videos"): image_url = snap["videos"][0].get("video_preview_image_url")
                if not image_url: continue

                ad_id = f"RAD{existing_count + 1:04d}"
                img_path = os.path.join(IMAGES_DIR, f"{ad_id}.jpg")
                try:
                    download_image(image_url, img_path)
                    description = generate_image_description(img_path)
                    new_row = {"Sample ID": ad_id, "Advertisement Type": detect_ad_type(clean_text), "Dialect": dialect_name, "Text Content": clean_text, "Image Description": description, "Metadata": f"page: {page_name} | date: {collection_date} | country: {country_code}", "Image File": f"{ad_id}.jpg", "Label": "Legitimate"}
                    with open(CSV_PATH, "a", newline="", encoding="utf-8-sig") as f: csv.DictWriter(f, fieldnames=fieldnames).writerow(new_row)
                    existing_texts.add(clean_text); dialect_counts[dialect_name] += 1; country_counts[country_code] = country_counts.get(country_code, 0) + 1; existing_count += 1; batch_added += 1; session_saved += 1
                    print_status(existing_count, dialect_counts, session_saved, f"saved {ad_id}")
                except: continue

            if batch_added == 0:
                empty_rounds += 1
                if empty_rounds >= 3: break
            else: empty_rounds = 0
            group_idx += 1

# ===========================================================
#  Completion Report
# ===========================================================
final_pct = existing_count / TOTAL_TARGET * 100
log(f"SESSION DONE | {existing_count}/2000 ({final_pct:.1f}%) | session_adds={session_saved} | tokens_used={current_token_idx+1}", "DONE")
print_status(existing_count, dialect_counts, session_saved, "SESSION COMPLETE")
if os.path.exists(CSV_PATH):
    display(pd.read_csv(CSV_PATH).tail(5))
